# Launching multiple agenst in parallel

import libraries

In [16]:
import json
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
import anthropic

import os
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) 

In [18]:
# function to parse JSON from a string, handling code blocks and extracting the JSON content
def _parse_json(text: str) -> dict:
    text = text.strip()
    m = re.search(r'```(?:json)?\s*\n?([\s\S]+?)\n?\s*```', text)
    if m:
        text = m.group(1).strip()
        start, end = text.find('{'), text.rfind('}')+1
    if start != -1 and end > start:
        text = text[start:end]
    return json.loads(text)

Define the handoff packets

In [19]:
@dataclass
class HandoffPacket:
    context: str
    task: str

def build_financial_handoff(company: str, fiscal_year: int) -> HandoffPacket:
    context = f"Due diligence for {company}, fiscal year {fiscal_year}."
    task = (
        f"Summarize the financial profile of {company} for FY {fiscal_year} based on "
        "publicly available information. Return a JSON object with these keys: "
        "revenue_estimate (string or null), growth_signal ('positive'/'neutral'/'negative'), "
        "and risk_signal (list of strings), data_gaps (list of strings), sources (list of strings). "
        "Return only the JSON object."
    )
    return HandoffPacket(context=context, task=task)

def build_legal_handoff(company: str) -> HandoffPacket:
    context = f"Legal Compliance for {company}." 
    task = (
        f"Access the regulatory and legal standing of {company} based on publicly available "
        "information. Return a JSON object with these keys: "
        "compliance_signal ('clean'/'caution'/'unknown'), nores (list of strings), "
        "data_gaps (list of strings), sources (list of strings). "
        "Do not invent specific case numbers. Return only the JSON object."
    )
    return HandoffPacket(context=context, task=task)

def build_news_handoff(company: str, days: int = 90) -> HandoffPacket:
    context = f"News sentiment for {company}, last {days}." 
    task = (
        f"Summarize public news sentiment for company {company} over the past {days} days. "
        "Return only the JSON object with these keys: "
        "sentiment ('positive'/'neutral'/'negative'), highlights (list of up to 3 strings), "
        "data_gaps (list of strings), sources (list of strings). "
        "Return only the JSON object."
    )
    return HandoffPacket(context=context, task=task)
    
def packet_to_prompt(packet: HandoffPacket) -> str:
    return f"Context: {packet.context}\nTask: {packet.task}"



In [20]:
# Define the specialists
SPECIALIST_SYSTEMS = {
    "financial": "You are a financial due diligence specialist. Return structured JSON only.",
    "legal": "You are a legal compliance specialist. Return structured JSON only.",
    "news": "You are a news sentiment analyst. Return structured JSON only."
}

model =  "claude-sonnet-5-5"
def run_specialist(system: str, prompt:str) -> dict:
    response = client.completions.create(
        model=model,
        system=system,
        max_tokens=1024,
        messages=[{"role":"user","content":prompt}]
    )
    for block in response.content:
        if block.type == "text":
            return _parse_json(block.text)
    return {"error":"no_text_block", "data_gaps":["specialist returned no content"]}

def dispatch_all_specialists(company: str, fiscal_year: int) -> dict:
    handoffs = {}
    handoffs["financial"] = build_financial_handoff(company, fiscal_year)
    handoffs["legal"] = build_legal_handoff(company)
    handoffs["news"] = build_news_handoff(company)

    results ={}
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {
            executor.submit(
                run_specialist,
                SPECIALIST_SYSTEMS[name],
                packet_to_prompt(packet)
            ): name
            for name, packet in handoffs.items()
        }
        for future in as_completed(futures):
            name = futures[future]
            try:
                results[name] = future.result()
            except Exception as e:
                results[name] = {
                    "error": "Specilaist_exception",
                    "message": str(e),
                    "data_gaps": [f"All {name} data unavailable, specialist failed to complete"]
                }
    return results

EXPECTED_SPECIALISTS = {"financial", "legal", "news"}


In [21]:
def validate_results(results: dict) -> tuple[dict, list[str]]:
    gaps = []
    validated = {}
    for name in EXPECTED_SPECIALISTS:
        if name not in results:
            gaps.append(f"{name} specialist did not return results")
            validated[name] = None
            continue
        result = results[name]
        if "error" in result: 
            gaps.append(f"{name} specilaist: {result.get('message', 'unknown error')}")
        validated[name] = result
    return validated, gaps

def synthesize_results(results: dict, report_gaps:list[str]) -> dict:
    merged_findings = {}
    aggregated_gaps = list(report_gaps)
    conflict_flags = []

    for name, result in validated.items():
        if result is None:
            aggregated_gaps.append(f"{name}: no data returned")
            continue
        for key, value in result.items():
            if key == "data_gaps":
                aggregated_gaps.extend(value)
            elif key in ("error", "message"):
                pass  # ignore error and message keys in synthesis
            else:
                if key in merged_findings:
                    conflict_flags.append(
                        f"Conflicting values for '{key}': "
                        f"{merged_findings[key]!r} vs {value!r} from {name})"
                    )
                else:
                    merged_findings[key] = value
    return {
        "findings":    merged_findings,
        "gaps":        aggregated_gaps,
        "conflicts":   conflict_flags,
        "is_complete": len(aggregated_gaps) == 0 and len(conflict_flags) == 0,
    }

In [22]:
COMPANY     = "Stripe"
FISCAL_YEAR = 2025

print(f"=== Parallel specialist dispatch: {COMPANY} FY{FISCAL_YEAR} ===\n")
print("Dispatching all three specialists simultaneously...\n")

raw = dispatch_all_specialists(COMPANY, FISCAL_YEAR)

print("All specialists have returned their results. Validating...\n")
for name, result in raw.items():
    status = "error" if "error" in result else "ok"
    keys   = [k for k in result if k not in ("data_gaps", "sources", "error", "message")]
    print(f"  {name:<12} [{status}]  finding keys: {keys}")

print("\n=== Validation ===\n")
validated, gaps = validate_results(raw)
print(f"Validation gaps : {gaps or 'none'}")

print("\n=== Synthesis ===\n")
report = synthesize_results(validated, gaps)

print("Merged findings:")
for key, value in report["findings"].items():
    display = str(value)
    if len(display) > 80:
        display = display[:80] + "..."
    print(f"  {key:<22} : {display}")

print(f"\nAggregated gaps : {report['gaps'] or 'none'}")
print(f"Conflicts       : {report['conflicts'] or 'none'}")
print(f"is_complete     : {report['is_complete']}")

=== Parallel specialist dispatch: Stripe FY2025 ===

Dispatching all three specialists simultaneously...

All specialists have returned their results. Validating...

  financial    [error]  finding keys: []
  news         [error]  finding keys: []
  legal        [error]  finding keys: []

=== Validation ===

Validation gaps : ["financial specilaist: 'Anthropic' object has no attribute 'completions'", "news specilaist: 'Anthropic' object has no attribute 'completions'", "legal specilaist: 'Anthropic' object has no attribute 'completions'"]

=== Synthesis ===

Merged findings:

Aggregated gaps : ["financial specilaist: 'Anthropic' object has no attribute 'completions'", "news specilaist: 'Anthropic' object has no attribute 'completions'", "legal specilaist: 'Anthropic' object has no attribute 'completions'", 'All financial data unavailable, specialist failed to complete', 'All news data unavailable, specialist failed to complete', 'All legal data unavailable, specialist failed to complet